# Satellite Geometry: Positions, Elevation, and Azimuth

**Version:** 1.0 | **Last updated:** 2026-07-31

**Author:** Eshanta Mishra | **Author institution:** EarthScope Consortium

**Maintainer:** EarthScope OnRamp Team | **Maintainer's contact:** help@earthscope.org

**Estimated Time:** ~ 40 minutes | **Pathway:** MVP1

**License:** CC-BY-4.0

## Introduction

**What this notebook does:** It retrieves GNSS satellite positions with the EarthScope SDK, computes the elevation and azimuth of each satellite as seen from a ground station, and joins that geometry to the observations recorded at the same station.

**Why it is useful:** An observation on its own tells us the strength of a signal but not where that signal came from. Satellite ephemeris positions supply the direction, so each measurement can be related to the satellite position in the sky. Once each measurement is tagged with an elevation and an azimuth, we can see how signal quality varies with position in the sky, which is the basis for elevation masks, multipath assessment, and GNSS reflectometry.

**What you will accomplish:** By the end of this notebook, we will produce a sky plot for a GNSS station showing how signal strength varies across the visible hemisphere, built from a dataframe in which every observation carries the elevation and azimuth of the satellite that produced it.

---

### Prerequisites

Before starting this notebook, you should:
* [ ] Have completed: [Notebook 1 - Accessing GNSS Observations with the EarthScope SDK](NB1-access-gnss-via-SDK.ipynb).
* [ ] Be familiar with basic python.

---

### GeoLab Compute Resources

| Setting | Recommended |
|---|---|
| **Image** | GeoLab (default image)|
| **Server size** | 4 GB RAM, ~0.5 CPUs (default server) |

## Learning Objectives

By the end of this notebook, you will be able to:

1. Retrieve satellite ephemeris positions for selected constellations and time ranges
2. Compute elevation and azimuth to each satellite from a reference station location
3. Join satellite geometry to GNSS observations on timestamp, satellite, and system
4. Visualize signal strength as a function of elevation and azimuth

## Relevant Documentation & Resources

* [EarthScope SDK documentation](https://docs.earthscope.org/sdk)
* [SDK GNSS Satellite Ephemeris Positions tutorial](https://docs.earthscope.org/sdk/gnss-eph-pos-tutorial)
* [Plotly](https://plotly.com/python/)
* [Altair](https://altair-viz.github.io/)

## Contents

1. [What is Satellite Geometry?](#id-1-what-is-satellite-geometry)
2. [Setup & Imports](#id-2-setup-imports)
3. [Retrieve Satellite Positions](#id-3-retrieve-satellite-positions)
4. [Visualize Satellite Orbits](#id-4-visualize-satellite-orbits)
5. [Elevation and Azimuth from a Station](#id-5-elevation-and-azimuth-from-a-station)
6. [Joining Geometry with Observations](#id-6-joining-geometry-with-observations)
7. [SNR vs Elevation](#id-7-snr-vs-elevation)
8. [Polar Sky Plot](#id-8-polar-sky-plot)
9. [Exploration Exercises](#id-9-exploration-exercises)
10. [Troubleshooting & Support](#id-10-troubleshooting-support)

## 1. What is Satellite Geometry?

Every GNSS observation is a measurement of a signal that travelled from a satellite to a receiver.Where that satellite was at the moment of the measurement is not part of the observation record, but it is known because the satellites broadcast their own orbital information. EarthScope provides these positions as a separate data product within the SDK.

**Ephemeris positions** are satellite locations given in an Earth-Centered, Earth-Fixed (ECEF) frame. The origin sits at the center of the Earth, and the axes rotate with the planet, so a fixed point on the ground keeps the same coordinates all day. Positions are in meters, as `x`, `y`, and `z`.

ECEF is convenient for computation but hard to interpret because the coordinates alone do not tell us whether a satellite was overhead or near the horizon. For that, the position has to be expressed relative to a particular place on the ground:

1. **Elevation**: It is the angle above the local horizon, from 0 degrees at the horizon to 90 degrees directly overhead. Negative values mean the satellite is below the horizon and not visible.
2. **Azimuth**: It is the compass direction to the satellite, measured in degrees clockwise from north, so 0 is north, 90 is east, 180 is south, and 270 is west.

Together, elevation and azimuth place a satellite on the dome of sky above a station. The SDK will compute both if you give it a reference point, so you do not have to do the coordinate conversion yourself or get separate orbit files such as when working with RINEX data.

**Why geometry matters for the observations.** A signal from a satellite near the horizon travelsa longer path through the atmosphere and arrives at a shallower angle, where it is more easily obstructed and more likely to reach the antenna after bouncing off the ground or a nearby surface. Signals from high overhead do not have these problems to the same degree. Elevation is therefore one of the strongest predictors of signal quality, and Sections 7 and 8 show that relationship directly.

## 2. Setup & Imports

In [ ]:
import datetime as dt

import altair as alt
import numpy as np
import plotly.colors as pcolors
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

from earthscope_sdk import AsyncEarthScopeClient
from earthscope_sdk.client.data_access.models import (
    FloatFilter,
    GeodeticCoordinate,
    SatelliteSystem,
)

alt.data_transformers.enable("vegafusion")  # Enable rust backend for altair
pio.renderers.default = "notebook"          # Required for plotly to render in the notebook

es = AsyncEarthScopeClient()

### Configuration

In [ ]:
STATION = "P041"                          # GNSS station (4-character ID)
SESSION = "A"                             # Session name
START = dt.datetime(2025, 9, 22)          # Query start (UTC)
END = dt.datetime(2025, 9, 23)            # Query end (UTC)

SYSTEM = "G"                              # Constellation: GPS
SATELLITES = [5, 7, 11, 13]               # Satellite numbers to follow
OBS_CODE = "1C"                           # Signal: GPS L1 C/A

# The reference point on the ground, used to compute elevation and azimuth.
STATION_LOCATION = GeodeticCoordinate(
    latitude=39.7392358,
    longitude=-104.990251,
    height=1000,
)

ORBIT_INTERVAL = dt.timedelta(minutes=5)  # Sample spacing for the orbit plot
AZEL_INTERVAL = dt.timedelta(seconds=15)  # Sample spacing for elevation and azimuth

> **Note:** `AZEL_INTERVAL` is 15 seconds because that is the sampling interval of the observations at this station. Section 6 joins the two datasets on their timestamps, and a join only matches rows whose timestamps agree exactly. If you change one of these intervals, the other has to follow.

## 3. Retrieve Satellite Positions

**What:** Satellite positions in the ECEF frame, for every constellation, sampled every five minutes across one day, retrieved with `gnss_ephemeris_positions()`.

**Why:** Five-minute sampling is coarse for analysis but ample for drawing orbits, and it keeps the request small. Orbits are smooth, so a satellite's path is well described by relatively few points.

**Expected result:** A dataframe of `timestamp`, `system`, `satellite`, `x`, `y`, `z`, with 32,392 rows. Coordinates are in meters.

In [ ]:
table = await es.data.gnss_ephemeris_positions(
    start_datetime=START,
    end_datetime=END,
    field=["x", "y", "z"],
    system=["G", "R", "E", "C", "S"],
    sample_interval=ORBIT_INTERVAL,
).fetch()

orbits = pl.from_arrow(table).sort("timestamp")
print(f"{len(orbits):,} rows")
orbits.head()

> **Check:** The `system` column should contain the constellation codes introduced in Notebook 1: `G` for GPS, `R` for GLONASS, `E` for Galileo, `C` for BeiDou, and `S` for SBAS. An interesting observation could be that this request has no `station_name`: satellite positions do not depend on who is observing them.

## 4. Visualize Satellite Orbits

The function below plots satellite trajectories in the ECEF frame, with a wireframe ellipsoid at
the Earth's own radii for scale. Each satellite is drawn as a separate line, so one constellation
produces a set of interleaved orbital tracks.

In [ ]:
def plot_satellite_positions(df: pl.DataFrame, system: SatelliteSystem):
    """
    Plot satellite positions in ECEF reference frame with an earth-sized ellipsoid.
    """
    # Group coordinates by svid
    coords = df.filter(pl.col("system") == system).to_dicts()
    coords_by_svid = {}
    for coord in coords:
        svid = coord["satellite"]
        if svid not in coords_by_svid:
            coords_by_svid[svid] = {"x": [], "y": [], "z": [], "timestamp": []}
        coords_by_svid[svid]["x"].append(coord["x"])
        coords_by_svid[svid]["y"].append(coord["y"])
        coords_by_svid[svid]["z"].append(coord["z"])
        coords_by_svid[svid]["timestamp"].append(coord["timestamp"])

    # Earth's radii in meters (WGS84)
    equatorial_radius = 6378137.0
    polar_radius = 6356752.3

    # Create the Earth's ellipsoid wireframe
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x_earth = equatorial_radius * np.outer(np.cos(u), np.sin(v))
    y_earth = equatorial_radius * np.outer(np.sin(u), np.sin(v))
    z_earth = polar_radius * np.outer(np.ones(np.size(u)), np.cos(v))

    earth_wireframe = go.Surface(
        x=x_earth,
        y=y_earth,
        z=z_earth,
        colorscale="Blues",
        showscale=False,
        opacity=0.2,
        name="Earth",
    )

    # Create a 3D scatter plot for each satellite trajectory
    satellite_traces = []
    colors = pcolors.qualitative.Plotly
    for i, (svid, coords_list) in enumerate(coords_by_svid.items()):
        satellite_traces.append(
            go.Scatter3d(
                x=coords_list["x"],
                y=coords_list["y"],
                z=coords_list["z"],
                mode="lines",
                text=coords_list["timestamp"],
                hoverinfo="all",
                line=dict(color=colors[i % len(colors)], width=2),
                name=f"SVID {svid}",
            )
        )

    fig = go.Figure(data=[earth_wireframe] + satellite_traces)

    # Update the layout for a cleaner look
    fig.update_layout(
        title="Satellite XYZ Coordinates (System: {})".format(system),
        scene=dict(
            xaxis_title="X Coordinate",
            yaxis_title="Y Coordinate",
            zaxis_title="Z Coordinate",
            aspectmode="data",
        ),
        margin=dict(r=20, b=10, l=10, t=40),
    )

    return fig

In [ ]:
fig = plot_satellite_positions(orbits, SYSTEM)
fig.show(height=800)

**What to look for:** The orbits form a shell well outside the Earth ellipsoid, at a radius of roughly four Earth radii. The tracks sit at several distinct inclinations rather than all in one plane, which is what keeps several satellites visible from any point on the ground at any time.

The plot is interactive. Drag to rotate, scroll to zoom, and hover a track to read its timestamp. Change `SYSTEM` in the Configuration cell to `"R"`, `"E"`, or `"C"` and re-run this cell to compare constellations.

## 5. Elevation and Azimuth from a Station

**What:** The same ephemeris product, but requesting `elevation` and `azimuth` instead of `x`, `y`, and `z`, and supplying a `reference_point` on the ground.

**Why:** Elevation and azimuth are what relate a satellite to a particular observer. Passing `reference_point` moves that computation to the server, so you receive angles directly rather than converting ECEF coordinates yourself.

**Expected result:** A dataframe of `timestamp`, `system`, `satellite`, `elevation`, `azimuth`, sampled every 15 seconds for the four satellites named in Configuration. Elevation and azimuth are in degrees.

In [ ]:
table = await es.data.gnss_ephemeris_positions(
    start_datetime=START,
    end_datetime=END,
    field=["elevation", "azimuth"],
    system=SYSTEM,
    satellite=SATELLITES,
    reference_point=STATION_LOCATION,
    sample_interval=AZEL_INTERVAL,
    elevation_filter=FloatFilter(min=0),
).fetch()

azel = pl.from_arrow(table).sort("timestamp")
print(f"{len(azel):,} rows")
azel.head()

`elevation_filter=FloatFilter(min=0)` discards every epoch at which a satellite was below the
horizon. Those rows describe geometry that no receiver could have observed, so filtering them
server-side keeps the response smaller. Raise the minimum to apply an elevation mask: `min=10`
returns only satellites at least 10 degrees above the horizon.

> **Check:** Every `elevation` value should be between 0 and 90, and every `azimuth` between 0 and 360.

## 6. Joining Geometry with Observations

The geometry and the observations are two separate products. Bringing them together is what lets
you ask how signal strength varies with position in the sky.

First, retrieve the observations, using the same station, satellites, and window.

In [ ]:
table = await es.data.gnss_observations(
    start_datetime=START,
    end_datetime=END,
    station_name=STATION,
    session_name=SESSION,
    system=SYSTEM,
    satellite=SATELLITES,
    obs_code=OBS_CODE,
    field="snr",
).fetch()

obs = pl.from_arrow(table).sort("timestamp")
print(f"{len(obs):,} rows")
obs.head()

The two dataframes share three columns that together identify a measurement: `timestamp`, `satellite`, and `system`. Joining on all three matches each observation to the geometry of the satellite that produced it, at the moment it was recorded.

An inner join keeps only rows that matched on both sides. Rows that fail to match are dropped: observations at epochs where the geometry was filtered out, and geometry for epochs where the receiver recorded nothing.

In [ ]:
joined = obs.join(
    azel,
    on=["timestamp", "satellite", "system"],
    how="inner",
).sort("timestamp")

print(f"observations: {len(obs):,}")
print(f"geometry:     {len(azel):,}")
print(f"joined:       {len(joined):,}")
joined.head()

> **Check:** The joined row count should be close to, but slightly below, both inputs. A joined count of zero means the timestamps never matched, which usually means `AZEL_INTERVAL` does not equal the observation sampling interval. A count far below both inputs means the two datasets overlap only partially.

## 7. SNR vs Elevation

With geometry attached to every observation, signal strength can be plotted against position in the sky rather than against time. The plot below restricts to elevations under 30 degrees, where the relationship is steepest.

In [ ]:
joined.filter(pl.col("elevation") < 30).with_columns(
    pl.col("satellite").cast(pl.Utf8)
).plot.point(x="elevation", y="snr", color="satellite").properties(
    title=f"{STATION}: SNR by elevation",
    width=650,
    height=300,
)

**What to look for:** SNR rises steeply as elevation increases from the horizon, then flattens. Most of the variation is concentrated in the lowest few degrees, which is why elevation masks are usually set low rather than high.

The oscillation near the horizon is worth attention. Instead of climbing smoothly, SNR ripples up and down with a wavelength of a degree or two. This is interference between the signal arriving directly from the satellite and the same signal arriving after reflecting off the ground nearby. The two paths differ in length by an amount that changes as the satellite rises, so they alternate between reinforcing and canceling.

That ripple is the signal GNSS reflectometry works from. Its frequency depends on the height of the antenna above the reflecting surface, so measuring the ripple lets you infer that height, and changes in it over time.

## 8. Polar Sky Plot

A sky plot maps the whole visible hemisphere onto a disc. Azimuth runs around the circle, and elevation runs along the radius with the zenith at the center and the horizon at the rim. Each satellite's pass appears as a track across the sky, colored here by mean SNR.

In [ ]:
(
    alt.Chart(joined)
    .mark_arc(tooltip=True)
    .encode(
        alt.Theta("azimuth:Q", bin=alt.Bin(maxbins=180), scale=alt.Scale(domain=[0, 360])),
        alt.Radius("elevation:Q", bin=alt.Bin(step=2), scale=alt.Scale(domain=[90, 0])),
        alt.Color("mean(snr):Q", title="Mean SNR", scale=alt.Scale(scheme="viridis")),
    )
    .properties(
        title=f"{STATION}: mean SNR by elevation and azimuth",
        width=500,
        height=500,
    )
)

**What to look for:** Read the disc as the sky seen from directly below, looking up. The center is overhead; the rim is the horizon. 

Tracks are brightest near the center and dimmest at the rim, which is the elevation relationship from Section 7 in a different projection. What the sky plot adds is direction. If some part of the rim is consistently darker than the rest at the same elevation, the sky is obstructed in that direction, and the azimuth tells you which way to look for the cause.

At mid-northern latitudes the middle of the northern sky is usually empty. GPS orbits are inclined about 55 degrees, so from this station no satellite passes directly through the far north. That gap is expected and is a property of the orbits, not of the station.

## 9. Exploration Exercises

Now that you've completed the core workflow, try modifying the parameters below to explore how the results change.

**Try these modifications:**

1. **Compare constellations:** Change `SYSTEM` in the Configuration section to `"R"`, `"E"`, or `"C"` and re-run Section 4. How do the orbital inclinations differ? Try `"S"` as well: SBAS satellites are geostationary, and their tracks look different.

2. **Apply an elevation mask:** Change `elevation_filter=FloatFilter(min=0)` in Section 5 to `min=10` or `min=15` and re-run Sections 5 to 8. How many observations does the join lose, and what happens to the rim of the sky plot?

In [ ]:
# Exploration cell — use this space to experiment

## 10. Troubleshooting & Support

### Common Issues

| Error | Likely cause | Fix |
|---|---|---|
| Joined dataframe has 0 rows | `AZEL_INTERVAL` does not match the observation sampling interval, so no timestamps align | Set `AZEL_INTERVAL` to the observation interval, 15 seconds at this station |
| All `elevation` values are negative or the result is empty | The reference point is on the far side of the Earth from these satellites, or latitude and longitude were swapped | Check `STATION_LOCATION`: latitude runs -90 to 90, longitude -180 to 180 |
| Sky plot is nearly empty | Only a few satellites were requested, or the elevation filter is aggressive | Add entries to `SATELLITES`, or lower the `elevation_filter` minimum |

### Further Resources

* [EarthScope SDK Documentation](https://docs.earthscope.org/sdk)
* [SDK GNSS Satellite Ephemeris Positions tutorial](https://docs.earthscope.org/sdk/gnss-eph-pos-tutorial)
* [SDK GNSS Observations tutorial](https://docs.earthscope.org/sdk/gnss-obs-tutorial)
* [GeoLab Documentation](https://docs.earthscope.org/geolab)
* [GeoLab Community Forum](https://earthscope.discourse.group/latest)